# Data Cleaning & Preparation

This notebook focuses on cleaning and preparing the raw Indonesia e-commerce dataset for exploratory data analysis, SQL analysis, and dashboard development.

The cleaning process includes:

- Standardizing column names
- Correcting data types
- Investigating and handling missing values
- Standardizing categorical variables
- Validating transaction consistency
- Investigating potential outliers
- Creating analytical features
- Performing final data quality checks

The original raw dataset is preserved and all transformations are performed on a separate working copy.

## Import Libraries

In [10]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## Load Raw Dataset

In [11]:
file_path = "../data/raw/all_months_clean.csv"

df_raw = pd.read_csv(file_path, sep=";")

# Create a working copy
df = df_raw.copy()

print("Raw dataset loaded successfully!")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Raw dataset loaded successfully!
Rows    : 20,848
Columns : 19


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20848 entries, 0 to 20847
Data columns (total 19 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   order_id                            20848 non-null  object
 1   total_qty                           20848 non-null  int64 
 2   total_weight_gr                     20848 non-null  int64 
 3   total_returned_qty                  20848 non-null  int64 
 4   Total Diskon                        20848 non-null  int64 
 5   product_categories                  20848 non-null  object
 6   num_product_categories              20848 non-null  int64 
 7   Status Pesanan                      20848 non-null  object
 8   Alasan Pembatalan                   2830 non-null   object
 9   Opsi Pengiriman                     20848 non-null  object
 10  Metode Pembayaran                   20848 non-null  object
 11  Kota/Kabupaten                      20848 non-null  ob

## Standardize Column Names

In [13]:
df.columns = [
    "order_id",
    "total_quantity",
    "total_weight_gram",
    "total_returned_quantity",
    "total_discount",
    "product_categories",
    "num_product_categories",
    "order_status",
    "cancellation_reason",
    "shipping_option",
    "payment_method",
    "city_regency",
    "province",
    "buyer_shipping_cost",
    "estimated_shipping_discount",
    "total_payment",
    "estimated_shipping_cost",
    "order_created_at",
    "source_file"
]

df.columns.tolist()

['order_id',
 'total_quantity',
 'total_weight_gram',
 'total_returned_quantity',
 'total_discount',
 'product_categories',
 'num_product_categories',
 'order_status',
 'cancellation_reason',
 'shipping_option',
 'payment_method',
 'city_regency',
 'province',
 'buyer_shipping_cost',
 'estimated_shipping_discount',
 'total_payment',
 'estimated_shipping_cost',
 'order_created_at',
 'source_file']

### Column Name Standardization

Column names were standardized using lowercase `snake_case` formatting.

This transformation improves readability and consistency across Python, SQL, and Power BI while avoiding issues caused by spaces, capitalization differences, and special characters.

The column names were also translated into consistent English terminology to make the project easier to document and present in a professional portfolio.

## Correct Data Types

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20848 entries, 0 to 20847
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   order_id                     20848 non-null  object
 1   total_quantity               20848 non-null  int64 
 2   total_weight_gram            20848 non-null  int64 
 3   total_returned_quantity      20848 non-null  int64 
 4   total_discount               20848 non-null  int64 
 5   product_categories           20848 non-null  object
 6   num_product_categories       20848 non-null  int64 
 7   order_status                 20848 non-null  object
 8   cancellation_reason          2830 non-null   object
 9   shipping_option              20848 non-null  object
 10  payment_method               20848 non-null  object
 11  city_regency                 20848 non-null  object
 12  province                     20848 non-null  object
 13  buyer_shipping_cost          20

from Data understanding before we know that order_created_at = object ❌

In [15]:
df["order_created_at"].head(10)

0    2024-04-01 00:15
1    2024-04-01 01:47
2    2024-04-01 04:25
3    2024-04-01 04:41
4    2024-04-01 06:12
5    2024-04-01 07:09
6    2024-04-01 07:23
7    2024-04-01 07:33
8    2024-04-01 07:56
9    2024-04-01 08:27
Name: order_created_at, dtype: object

In [16]:
df["order_created_at"] = pd.to_datetime(
    df["order_created_at"],
    errors="coerce"
)

df["order_created_at"].dtype

dtype('<M8[ns]')

In [17]:
print(
    "Missing datetime after conversion:",
    df["order_created_at"].isna().sum()
)

Missing datetime after conversion: 1980


### Datetime Conversion

`order_created_at` was originally stored as an object even though it represents transaction timestamps.

The column was converted to Pandas `datetime64` format to enable time-based analysis such as monthly trends, yearly comparisons, and seasonality analysis.

Invalid values were converted to missing values using `errors="coerce"`. The number of missing timestamps remained unchanged after conversion, indicating that the conversion process did not introduce additional missing values.

## Investigate Missing Cancellation Reasons

In [18]:
pd.crosstab(
    df["order_status"],
    df["cancellation_reason"].isna(),
    margins=True
)

cancellation_reason,False,True,All
order_status,,,
Batal,2830,0,2830
"Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-05.",0,25,25
"Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-06.",0,34,34
"Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-07.",0,34,34
"Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-08.",0,38,38
"Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-09.",0,22,22
"Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-10.",0,6,6
"Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-11.",0,2,2
Sedang Dikirim,0,59,59


In [19]:
df.groupby("order_status").agg(
    total_orders=("order_id", "count"),
    missing_cancellation_reason=(
        "cancellation_reason",
        lambda x: x.isna().sum()
    )
).sort_values("total_orders", ascending=False)

,total_orders,missing_cancellation_reason
order_status,,
Selesai,17768,17768
Batal,2830,0
Sedang Dikirim,59,59
"Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-08.",38,38
"Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-06.",34,34
"Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-07.",34,34
Telah Dikirim,30,30
"Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-05.",25,25
"Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-09.",22,22


In [20]:
df["order_status"].value_counts(dropna=False)

order_status
Selesai                                                                                   17768
Batal                                                                                      2830
Sedang Dikirim                                                                               59
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-08.       38
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-06.       34
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-07.       34
Telah Dikirim                                                                                30
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-05.       25
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-09.       22
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-10.        6
Pesanan diterima, namun Pem

In [21]:
df["cancellation_reason"].value_counts(dropna=False)

cancellation_reason
NaN                                                                                                                                                   18018
Dibatalkan oleh Pembeli. Alasan: Lainnya/ berubah pikiran                                                                                               662
Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan yang Ada                                                                                                  558
Dibatalkan secara otomatis oleh sistem. Alasan: Pesanan belum dibayar                                                                                   447
Dibatalkan oleh Pembeli. Alasan: Need to change delivery address                                                                                        306
Dibatalkan secara otomatis oleh sistem. Alasan: Pengiriman gagal                                                                                        255
Dibatalkan secara otomatis oleh sistem. Alas

### Cancellation Reason Investigation

The `cancellation_reason` column contains a large proportion of missing values. However, cancellation reasons are only expected to exist when an order is cancelled.

Therefore, the relationship between `order_status` and missing cancellation reasons was investigated before applying any treatment.

This prevents valid structural missing values from being incorrectly treated as data quality errors.

-------------

### Handling Missing Cancellation Reasons

The previous investigation confirmed that all cancelled orders contain a valid cancellation reason, while missing cancellation reasons occur exclusively in non-cancelled orders.

Therefore, the missing values do not represent unknown information. Instead, the cancellation reason is simply not applicable to those transactions.

The missing values are replaced with `Not Cancelled` to explicitly distinguish non-cancelled transactions from cancelled transactions with known reasons.

In [33]:
df["cancellation_reason"] = (
    df["cancellation_reason"]
    .fillna("Not Cancelled")
)

In [34]:
print(
    "Missing cancellation reasons:",
    df["cancellation_reason"].isna().sum()
)

print("\nCancellation reason distribution:")
print(df["cancellation_reason"].value_counts())

Missing cancellation reasons: 0

Cancellation reason distribution:
cancellation_reason
Not Cancelled                                                                                                                                         18018
Dibatalkan oleh Pembeli. Alasan: Lainnya/ berubah pikiran                                                                                               662
Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan yang Ada                                                                                                  558
Dibatalkan secara otomatis oleh sistem. Alasan: Pesanan belum dibayar                                                                                   447
Dibatalkan oleh Pembeli. Alasan: Need to change delivery address                                                                                        306
Dibatalkan secara otomatis oleh sistem. Alasan: Pengiriman gagal                                                                     

### Cancellation Reason Treatment Result

All **18,018 missing cancellation reasons** were identified as structurally missing values associated with non-cancelled orders.

These values were replaced with `Not Cancelled` rather than `Unknown`, because the investigation confirmed that every cancelled order already contains a recorded cancellation reason.

After treatment:

- Missing cancellation reasons: **0**
- Cancelled orders without a valid reason: **0**

This transformation preserves the business meaning of the variable while making it easier to use in SQL queries, dashboard filters, and cancellation analysis.

-----------------------

## Investigate Missing Order Timestamps

In [22]:
missing_date_by_source = (
    df[df["order_created_at"].isna()]
    .groupby("source_file")
    .size()
    .sort_values(ascending=False)
)

missing_date_by_source

source_file
DecemberSales2024.xlsx    1214
JulySales2025.xlsx         766
dtype: int64

In [23]:
source_summary = (
    df.groupby("source_file")
    .agg(
        total_rows=("order_id", "count"),
        missing_dates=("order_created_at", lambda x: x.isna().sum())
    )
)

source_summary["missing_percentage"] = (
    source_summary["missing_dates"]
    / source_summary["total_rows"]
    * 100
).round(2)

source_summary.sort_values(
    "missing_percentage",
    ascending=False
).head(10)

,total_rows,missing_dates,missing_percentage
source_file,,,
DecemberSales2024.xlsx,1214,1214,100.0
JulySales2025.xlsx,766,766,100.0
AugustSales2024.xlsx,1293,0,0.0
AprilSales2024.xlsx,492,0,0.0
AugustSales2025.xlsx,942,0,0.0
DecemberSales2023.xlsx,444,0,0.0
FebruarySales2024.xlsx,389,0,0.0
AprilSales2025.xlsx,824,0,0.0
FebruarySales2025.xlsx,957,0,0.0


In [24]:
df.loc[
    df["order_created_at"].isna(),
    ["order_id", "order_created_at", "source_file"]
].head(20)

,order_id,order_created_at,source_file
3995,ORD_0003996,NaT,DecemberSales2024.xlsx
3996,ORD_0003997,NaT,DecemberSales2024.xlsx
3997,ORD_0003998,NaT,DecemberSales2024.xlsx
3998,ORD_0003999,NaT,DecemberSales2024.xlsx
3999,ORD_0004000,NaT,DecemberSales2024.xlsx
4000,ORD_0004001,NaT,DecemberSales2024.xlsx
4001,ORD_0004002,NaT,DecemberSales2024.xlsx
4002,ORD_0004003,NaT,DecemberSales2024.xlsx
4003,ORD_0004004,NaT,DecemberSales2024.xlsx
4004,ORD_0004005,NaT,DecemberSales2024.xlsx


### Missing Timestamp Investigation

The missing timestamp analysis identified a systematic data quality issue rather than randomly distributed missing values.

All **1,980 missing order timestamps** originate from only two source files:

- `DecemberSales2024.xlsx`: **1,214 missing timestamps (100% of the file)**
- `JulySales2025.xlsx`: **766 missing timestamps (100% of the file)**

Other inspected source files do not contain missing timestamps.

This pattern indicates that the missing timestamps are likely related to a source-level data issue affecting entire monthly batches rather than individual transactions.

Dropping these records would remove approximately **9.5% of the dataset**, while imputing arbitrary dates would introduce fabricated information.

Therefore, these transactions will be retained. Reliable month and year information will instead be recovered from the `source_file` metadata for month-level analysis, while the exact transaction timestamp will remain missing.

------------

### Recovering Missing Time Information

Although the exact timestamps cannot be recovered, the source file names contain reliable month and year information.

Month-level temporal information will therefore be extracted from `source_file` without modifying or fabricating the original missing timestamps.

In [26]:
# Extract month name and year from source file
df["source_month_name"] = df["source_file"].str.extract(
    r"([A-Za-z]+)Sales"
)[0]

df["source_year"] = pd.to_numeric(
    df["source_file"].str.extract(r"(\d{4})")[0],
    errors="coerce"
)

df[[
    "source_file",
    "source_month_name",
    "source_year"
]].drop_duplicates().sort_values(
    ["source_year", "source_month_name"]
)

,source_file,source_month_name,source_year
3551,DecemberSales2023.xlsx,December,2023
0,AprilSales2024.xlsx,April,2024
1316,AugustSales2024.xlsx,August,2024
3995,DecemberSales2024.xlsx,December,2024
5209,FebruarySales2024.xlsx,February,2024
6555,JanuarySales2024.xlsx,January,2024
8136,JulySales2024.xlsx,July,2024
10172,JuneSales2024.xlsx,June,2024
11484,MarchSales2024.xlsx,March,2024
12565,MaySales2024.xlsx,May,2024


In [ ]:
# convert month to number
month_mapping = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}

df["source_month"] = (
    df["source_month_name"]
    .map(month_mapping)
    .astype("Int64")
)

df[[
    "source_file",
    "source_month",
    "source_year"
]].drop_duplicates().sort_values(
    ["source_year", "source_month"]
)

,source_file,source_month,source_year
3551,DecemberSales2023.xlsx,12,2023
6555,JanuarySales2024.xlsx,1,2024
5209,FebruarySales2024.xlsx,2,2024
11484,MarchSales2024.xlsx,3,2024
0,AprilSales2024.xlsx,4,2024
12565,MaySales2024.xlsx,5,2024
10172,JuneSales2024.xlsx,6,2024
8136,JulySales2024.xlsx,7,2024
1316,AugustSales2024.xlsx,8,2024
18594,SeptemberSales2024.xlsx,9,2024


### make analytical year/month column

In [ ]:
df["order_year"] = (
    df["order_created_at"]
    .dt.year
    .astype("Int64")
)

df["order_month"] = (
    df["order_created_at"]
    .dt.month
    .astype("Int64")
)

# Fill only missing year/month using source-file metadata
df["order_year"] = df["order_year"].fillna(
    df["source_year"]
)

df["order_month"] = df["order_month"].fillna(
    df["source_month"]
)

In [29]:
df["year_month"] = pd.to_datetime(
    df["order_year"].astype(str)
    + "-"
    + df["order_month"].astype(str)
    + "-01"
)

df[[
    "order_created_at",
    "source_file",
    "order_year",
    "order_month",
    "year_month"
]].loc[df["order_created_at"].isna()].head()

,order_created_at,source_file,order_year,order_month,year_month
3995,NaT,DecemberSales2024.xlsx,2024,12,2024-12-01
3996,NaT,DecemberSales2024.xlsx,2024,12,2024-12-01
3997,NaT,DecemberSales2024.xlsx,2024,12,2024-12-01
3998,NaT,DecemberSales2024.xlsx,2024,12,2024-12-01
3999,NaT,DecemberSales2024.xlsx,2024,12,2024-12-01


In [30]:
df["timestamp_available"] = (
    df["order_created_at"]
    .notna()
    .astype(int)
)

In [31]:
df["timestamp_available"].value_counts()

timestamp_available
1    18868
0     1980
Name: count, dtype: int64

In [32]:
print(
    "Missing exact timestamps:",
    df["order_created_at"].isna().sum()
)

print(
    "Missing analytical year:",
    df["order_year"].isna().sum()
)

print(
    "Missing analytical month:",
    df["order_month"].isna().sum()
)

print(
    "Missing year_month:",
    df["year_month"].isna().sum()
)

Missing exact timestamps: 1980
Missing analytical year: 0
Missing analytical month: 0
Missing year_month: 0


### Timestamp Treatment

The exact timestamps for 1,980 transactions could not be reliably recovered and were therefore left as missing (`NaT`).

However, month and year information were successfully recovered from the corresponding source-file names.

For transactions with valid timestamps, the actual timestamp was used to derive the analytical month and year. Source-file metadata was only used when the original timestamp was unavailable.

A `timestamp_available` flag was also created to preserve information about timestamp completeness.

This approach retains all transactions for monthly analysis while avoiding fabricated day-level or time-level information.

In [35]:
df.isna().sum()

order_id                          0
total_quantity                    0
total_weight_gram                 0
total_returned_quantity           0
total_discount                    0
product_categories                0
num_product_categories            0
order_status                      0
cancellation_reason               0
shipping_option                   0
payment_method                    0
city_regency                      0
province                          0
buyer_shipping_cost               0
estimated_shipping_discount       0
total_payment                     0
estimated_shipping_cost           0
order_created_at               1980
source_file                       0
source_month_name                 0
source_year                       0
source_month                      0
order_year                        0
order_month                       0
year_month                        0
timestamp_available               0
dtype: int64

## Missing Value Treatment Summary

After the missing value treatment, `cancellation_reason` no longer contains missing values.

The remaining **1,980 missing values in `order_created_at` are intentionally preserved** because the exact transaction date and time cannot be reliably reconstructed.

Instead, reliable month and year information was recovered from the corresponding source-file metadata and stored in `order_year`, `order_month`, and `year_month`.

Therefore:

- `cancellation_reason`: 0 missing values
- `order_year`: 0 missing values
- `order_month`: 0 missing values
- `year_month`: 0 missing values
- `order_created_at`: 1,980 missing values intentionally retained

A `timestamp_available` indicator is maintained to distinguish transactions with exact timestamps from transactions where only month-level temporal information is available.

This approach avoids fabricating transaction dates while preserving all 20,848 orders for analyses that do not require exact timestamps.

--------------------

## Categorical Data Standardization

Categorical variables are inspected and standardized to ensure that equivalent business categories are represented consistently.

This step focuses on identifying inconsistent labels, unnecessary variations, embedded dynamic information, and formatting differences that could distort grouping and aggregation during analysis.

### Clean order_status

In [36]:
df["order_status_raw"] = df["order_status"]

df["order_status_raw"].value_counts()

order_status_raw
Selesai                                                                                   17768
Batal                                                                                      2830
Sedang Dikirim                                                                               59
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-08.       38
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-06.       34
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-07.       34
Telah Dikirim                                                                                30
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-05.       25
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-09.       22
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-10.        6
Pesanan diterima, namun

In [37]:
df["order_status"] = (
    df["order_status"]
    .astype(str)
    .str.strip()
)

received_mask = df["order_status"].str.startswith(
    "Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga",
    na=False
)

df.loc[
    received_mask,
    "order_status"
] = "Pesanan Diterima"

In [38]:
print("Before standardization:")
print(df["order_status_raw"].nunique())

print("\nAfter standardization:")
print(df["order_status"].nunique())

print("\nStandardized order status:")
print(df["order_status"].value_counts())

Before standardization:
11

After standardization:
5

Standardized order status:
order_status
Selesai             17768
Batal                2830
Pesanan Diterima      161
Sedang Dikirim         59
Telah Dikirim          30
Name: count, dtype: int64


#### Order Status Standardization

The original `order_status` variable contained multiple labels representing the same business state because return-deadline dates were embedded directly in the status text.

These date-specific labels were standardized into a single `Pesanan Diterima` category.

The original values were preserved in `order_status_raw` to maintain data traceability.

This prevents identical business states from being treated as separate categories during aggregation, SQL analysis, and dashboard visualization.

----------------------

### make order_status_group column

In [39]:
df["order_status"].value_counts()

order_status
Selesai             17768
Batal                2830
Pesanan Diterima      161
Sedang Dikirim         59
Telah Dikirim          30
Name: count, dtype: int64

In [40]:
status_group_mapping = {
    "Selesai": "Completed",
    "Batal": "Cancelled",
    "Sedang Dikirim": "In Progress",
    "Telah Dikirim": "In Progress",
    "Pesanan Diterima": "Completed"
}

df["order_status_group"] = (
    df["order_status"]
    .map(status_group_mapping)
)

In [41]:
print(df["order_status_group"].value_counts(dropna=False))

print(
    "\nUnmapped status:",
    df["order_status_group"].isna().sum()
)

order_status_group
Completed      17929
Cancelled       2830
In Progress       89
Name: count, dtype: int64

Unmapped status: 0


#### Order Status Grouping Result

The standardized order statuses were grouped into broader business-level categories to simplify performance analysis and dashboard reporting.

The mapping produced three main order groups:

- **Completed:** 17,929 orders, consisting of `Selesai` and `Pesanan Diterima`.
- **Cancelled:** 2,830 orders originating from the `Batal` status.
- **In Progress:** 89 orders, consisting of `Sedang Dikirim` and `Telah Dikirim`.

All **20,848 orders were successfully mapped**, with **0 unmapped records**.

The original detailed order status is retained in `order_status`, while `order_status_group` provides a simplified business-level classification. This structure allows detailed operational analysis while also supporting high-level KPIs such as completion rate, cancellation rate, and orders currently in progress.

---------

### Payment Method Cleaning

In [42]:
print("Unique payment methods:")
print(df["payment_method"].nunique())

df["payment_method"].value_counts()

Unique payment methods:
12


payment_method
COD (Bayar di Tempat)        11538
Saldo ShopeePay               3692
Online Payment                3170
SPayLater                     1485
SeaBank Bayar Instan           637
Kartu Kredit/Debit             222
Indomaret/i.Saku                43
Alfamart/Alfamidi/Dan+Dan       36
Pembayaran dibebaskan           18
Cicilan Kartu Kredit             3
BCA OneKlik                      3
Mitra Shopee                     1
Name: count, dtype: int64

In [43]:
sorted(df["payment_method"].dropna().unique())

['Alfamart/Alfamidi/Dan+Dan',
 'BCA OneKlik',
 'COD (Bayar di Tempat)',
 'Cicilan Kartu Kredit',
 'Indomaret/i.Saku',
 'Kartu Kredit/Debit',
 'Mitra Shopee',
 'Online Payment',
 'Pembayaran dibebaskan',
 'SPayLater',
 'Saldo ShopeePay',
 'SeaBank Bayar Instan']

In [44]:
df["payment_method"] = (
    df["payment_method"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [45]:
df["payment_method"].value_counts()

payment_method
COD (Bayar di Tempat)        11538
Saldo ShopeePay               3692
Online Payment                3170
SPayLater                     1485
SeaBank Bayar Instan           637
Kartu Kredit/Debit             222
Indomaret/i.Saku                43
Alfamart/Alfamidi/Dan+Dan       36
Pembayaran dibebaskan           18
Cicilan Kartu Kredit             3
BCA OneKlik                      3
Mitra Shopee                     1
Name: count, dtype: int64

#### Payment Method Cleaning Result

The `payment_method` column contains **12 distinct payment methods**. After removing leading, trailing, and duplicated whitespace, no inconsistent or duplicated labels were identified.

The payment methods represent different transaction mechanisms, including Cash on Delivery (COD), digital wallets, online payments, bank-based payments, credit/debit cards, retail outlets, and credit installment services.

Because these categories contain meaningful business information, the original payment method categories were retained rather than merged.

The cleaned variable can later be used to analyze customer payment preferences, transaction volume by payment method, and potential relationships between payment methods and order outcomes.

--------------

### Shipping Option Cleaning

In [47]:
print("Unique shipping options:")
print(df["shipping_option"].nunique())

df["shipping_option"].value_counts()

Unique shipping options:
45


shipping_option
Hemat Kargo-SPX Hemat                                    12597
Reguler (Cashless)-SPX Standard                           4622
SPX Hemat                                                  543
Hemat Kargo                                                460
Reguler (Cashless)-JNE Reguler                             344
Instant (Versi Lama)-SPX Instant (Versi Lama)              336
Same Day-SPX Sameday                                       302
SPX Standard                                               192
Kargo-JNE Trucking (JTR)                                   190
Reguler (Cashless)                                         190
Hemat Kargo-J&T Economy                                    175
Kargo-J&T Cargo                                            150
Reguler (Cashless)-J&T Express                             106
Agen SPX Express                                            96
Hemat Kargo-J&T Cargo                                       72
J&T Cargo                              

In [48]:
df["shipping_option"] = (
    df["shipping_option"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [49]:
print(
    "Shipping options after whitespace cleaning:",
    df["shipping_option"].nunique()
)

Shipping options after whitespace cleaning: 45


#### Shipping Option Cleaning Result

The `shipping_option` column contains **45 unique values**. Basic whitespace normalization did not reduce the number of categories, indicating that the variation is not caused by formatting inconsistencies.

Further inspection shows that many shipping option labels contain a combination of **shipping service type** and **logistics provider**, such as regular, economy, cargo, same-day, and instant services provided by different couriers.

Therefore, the original `shipping_option` values are retained to preserve detailed logistics information.

Rather than merging these categories directly, additional analytical features such as `shipping_service` and `courier` can be derived later. This allows shipping performance to be analyzed at both the detailed service-option level and the broader logistics-provider level.

----------------------

### Province Cleaning

In [50]:
print("Number of provinces:", df["province"].nunique())

print(
    sorted(df["province"].dropna().unique())
)

Number of provinces: 34
['BALI', 'BANGKA BELITUNG', 'BANTEN', 'BENGKULU', 'DI YOGYAKARTA', 'DKI JAKARTA', 'GORONTALO', 'JAMBI', 'JAWA BARAT', 'JAWA TENGAH', 'JAWA TIMUR', 'KALIMANTAN BARAT', 'KALIMANTAN SELATAN', 'KALIMANTAN TENGAH', 'KALIMANTAN TIMUR', 'KALIMANTAN UTARA', 'KEPULAUAN RIAU', 'LAMPUNG', 'MALUKU', 'MALUKU UTARA', 'NANGGROE ACEH DARUSSALAM (NAD)', 'NUSA TENGGARA BARAT (NTB)', 'NUSA TENGGARA TIMUR (NTT)', 'PAPUA', 'PAPUA BARAT', 'RIAU', 'SULAWESI BARAT', 'SULAWESI SELATAN', 'SULAWESI TENGAH', 'SULAWESI TENGGARA', 'SULAWESI UTARA', 'SUMATERA BARAT', 'SUMATERA SELATAN', 'SUMATERA UTARA']


In [51]:
df["province"] = (
    df["province"]
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

In [52]:
print(
    "Number of provinces after cleaning:",
    df["province"].nunique()
)

print(
    sorted(df["province"].unique())
)

Number of provinces after cleaning: 34
['BALI', 'BANGKA BELITUNG', 'BANTEN', 'BENGKULU', 'DI YOGYAKARTA', 'DKI JAKARTA', 'GORONTALO', 'JAMBI', 'JAWA BARAT', 'JAWA TENGAH', 'JAWA TIMUR', 'KALIMANTAN BARAT', 'KALIMANTAN SELATAN', 'KALIMANTAN TENGAH', 'KALIMANTAN TIMUR', 'KALIMANTAN UTARA', 'KEPULAUAN RIAU', 'LAMPUNG', 'MALUKU', 'MALUKU UTARA', 'NANGGROE ACEH DARUSSALAM (NAD)', 'NUSA TENGGARA BARAT (NTB)', 'NUSA TENGGARA TIMUR (NTT)', 'PAPUA', 'PAPUA BARAT', 'RIAU', 'SULAWESI BARAT', 'SULAWESI SELATAN', 'SULAWESI TENGAH', 'SULAWESI TENGGARA', 'SULAWESI UTARA', 'SUMATERA BARAT', 'SUMATERA SELATAN', 'SUMATERA UTARA']


#### Province Cleaning Result

The `province` column contains **34 unique province values**.

Province names were standardized by removing leading and trailing whitespace, converting all values to uppercase, and normalizing repeated whitespace.

The number of unique provinces remained unchanged at **34** after standardization, indicating that no duplicated province categories were caused by inconsistent text formatting.

The cleaned province variable is now suitable for regional aggregation and geographic analysis in the later EDA and dashboard stages.

-------------------

### City/Regency Cleaning

In [53]:
df["city_regency"] = (
    df["city_regency"]
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

print(
    "Unique cities/regencies:",
    df["city_regency"].nunique()
)

df["city_regency"].value_counts().head(20)

Unique cities/regencies: 424


city_regency
KAB. BOGOR                1193
KAB. TANGERANG            1135
KOTA TANGERANG            1011
KAB. BEKASI                813
KOTA TANGERANG SELATAN     802
KOTA JAKARTA BARAT         729
KOTA JAKARTA SELATAN       695
KOTA JAKARTA TIMUR         680
KOTA BEKASI                623
KOTA DEPOK                 598
KOTA JAKARTA UTARA         491
KAB. BANDUNG               484
KAB. KARAWANG              435
KOTA BANDUNG               430
KOTA PALEMBANG             312
KOTA JAKARTA PUSAT         302
KOTA BOGOR                 245
KAB. SUKABUMI              222
KOTA BANDAR LAMPUNG        198
KAB. BANDUNG BARAT         188
Name: count, dtype: int64

In [54]:
df["city_regency"] = (
    df["city_regency"]
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

In [55]:
print(
    "Number of cities/regencies after cleaning:",
    df["city_regency"].nunique()
)

print("\nTop 20 after cleaning:")
print(df["city_regency"].value_counts().head(20))

Number of cities/regencies after cleaning: 424

Top 20 after cleaning:
city_regency
KAB. BOGOR                1193
KAB. TANGERANG            1135
KOTA TANGERANG            1011
KAB. BEKASI                813
KOTA TANGERANG SELATAN     802
KOTA JAKARTA BARAT         729
KOTA JAKARTA SELATAN       695
KOTA JAKARTA TIMUR         680
KOTA BEKASI                623
KOTA DEPOK                 598
KOTA JAKARTA UTARA         491
KAB. BANDUNG               484
KAB. KARAWANG              435
KOTA BANDUNG               430
KOTA PALEMBANG             312
KOTA JAKARTA PUSAT         302
KOTA BOGOR                 245
KAB. SUKABUMI              222
KOTA BANDAR LAMPUNG        198
KAB. BANDUNG BARAT         188
Name: count, dtype: int64


#### City/Regency Cleaning Result

The `city_regency` column contains **424 unique city and regency values**.

The values were standardized by removing leading and trailing whitespace, converting text to uppercase, and normalizing repeated whitespace.

The number of unique values remained unchanged at **424** after cleaning, indicating that no duplicated geographic categories were caused by inconsistent text formatting.

Administrative distinctions such as `KAB.` (Regency) and `KOTA` (City) were intentionally preserved because they represent different geographic entities.

The cleaned `city_regency` variable is now suitable for regional sales analysis and geographic visualization in the dashboard.+++

----------------------

### Categorical Standardization Summary

Categorical variables were reviewed and standardized to ensure consistency before further analysis.

The following preprocessing steps were performed:

- **Order Status:** Raw order status values were standardized and grouped into broader analytical categories: `Completed`, `Cancelled`, and `In Progress`.
- **Cancellation Reason:** Missing values were identified as structural missing values for non-cancelled orders and were handled accordingly while preserving actual cancellation reasons.
- **Payment Method:** Text formatting and whitespace were standardized while preserving the original payment method categories.
- **Shipping Option:** Formatting inconsistencies were cleaned. The 45 detailed shipping options were retained because they contain useful information about shipping services and logistics providers.
- **Province:** Province names were standardized using consistent capitalization and whitespace formatting. The dataset contains 34 unique provinces after cleaning.
- **City/Regency:** City and regency names were standardized while preserving administrative distinctions such as `KOTA` and `KAB.`. The dataset contains 424 unique city/regency values after cleaning.

Overall, categorical standardization improved consistency without unnecessarily removing or merging meaningful business information. The cleaned categorical variables are now suitable for exploratory data analysis, feature engineering, regional analysis, and dashboard development.

----------------

## Business Rule Validation

Business rules are applied to identify logically inconsistent or potentially invalid transactions. 
Unlike statistical outliers, these checks evaluate whether transaction values violate expected business relationships, such as returned quantities exceeding purchased quantities or negative monetary values.
Records identified during this stage are investigated before any removal or modification is performed.

### Validate quantity & return

In [56]:
business_rule_checks = {
    "quantity <= 0":
        (df["total_quantity"] <= 0).sum(),

    "returned quantity < 0":
        (df["total_returned_quantity"] < 0).sum(),

    "returned quantity > purchased quantity":
        (
            df["total_returned_quantity"]
            > df["total_quantity"]
        ).sum(),

    "weight <= 0":
        (df["total_weight_gram"] <= 0).sum(),

    "discount < 0":
        (df["total_discount"] < 0).sum(),

    "buyer shipping cost < 0":
        (df["buyer_shipping_cost"] < 0).sum(),

    "estimated shipping discount < 0":
        (df["estimated_shipping_discount"] < 0).sum(),

    "total payment < 0":
        (df["total_payment"] < 0).sum(),

    "estimated shipping cost < 0":
        (df["estimated_shipping_cost"] < 0).sum()
}

business_rule_summary = pd.DataFrame(
    business_rule_checks.items(),
    columns=["business_rule", "violations"]
)

business_rule_summary

,business_rule,violations
0,quantity <= 0,0
1,returned quantity < 0,0
2,returned quantity > purchased quantity,0
3,weight <= 0,0
4,discount < 0,0
5,buyer shipping cost < 0,0
6,estimated shipping discount < 0,0
7,total payment < 0,0
8,estimated shipping cost < 0,0


#### Basic Business Rule Validation Result

The transaction-level validation did not identify any violations in the basic numerical business rules.

All orders have positive purchased quantities and weights, returned quantities are non-negative and do not exceed purchased quantities, and no negative values were found in discount, payment, or shipping-related variables.

These results indicate that the core numerical transaction variables are logically consistent and do not require corrective treatment based on the evaluated business rules.

-------------

### Check relationship order status

In [57]:
status_checks = {
    "cancelled orders with returned items":
        (
            (df["order_status_group"] == "Cancelled") &
            (df["total_returned_quantity"] > 0)
        ).sum(),

    "completed orders with zero payment":
        (
            (df["order_status_group"] == "Completed") &
            (df["total_payment"] == 0)
        ).sum(),

    "completed orders with returned items":
        (
            (df["order_status_group"] == "Completed") &
            (df["total_returned_quantity"] > 0)
        ).sum()
}

status_validation = pd.DataFrame(
    status_checks.items(),
    columns=["business_rule", "records"]
)

status_validation

,business_rule,records
0,cancelled orders with returned items,0
1,completed orders with zero payment,62
2,completed orders with returned items,157


#### Order Status Validation Result

The relationship between order status, payment, and returned quantities was further investigated.

No cancelled orders contained returned quantities. However, **62 completed orders have a total payment of zero**, and **157 completed orders contain returned items**.

These records are not automatically considered invalid because they may represent legitimate business scenarios such as fully discounted transactions, refunds, or completed orders followed by product returns.

Therefore, these records require further investigation before determining whether any data treatment is necessary.

-------------------

### Investigate Completed Orders with Zero Payment

In [58]:
completed_zero_payment = df[
    (df["order_status_group"] == "Completed") &
    (df["total_payment"] == 0)
].copy()

print("Total records:", len(completed_zero_payment))

completed_zero_payment[
    [
        "order_id",
        "total_quantity",
        "total_discount",
        "buyer_shipping_cost",
        "estimated_shipping_discount",
        "estimated_shipping_cost",
        "total_payment",
        "payment_method",
        "order_status",
        "source_file"
    ]
].head(20)

Total records: 62


,order_id,total_quantity,total_discount,buyer_shipping_cost,estimated_shipping_discount,estimated_shipping_cost,total_payment,payment_method,order_status,source_file
236,ORD_0000237,2,0,0,0,8000,0,Saldo ShopeePay,Selesai,AprilSales2024.xlsx
900,ORD_0000901,1,0,0,0,12000,0,Online Payment,Selesai,AprilSales2025.xlsx
1797,ORD_0001798,1,0,0,0,24000,0,Saldo ShopeePay,Selesai,AugustSales2024.xlsx
1955,ORD_0001956,1,0,0,0,37500,0,COD (Bayar di Tempat),Selesai,AugustSales2024.xlsx
1970,ORD_0001971,70,0,0,0,18000,0,Online Payment,Selesai,AugustSales2024.xlsx
2735,ORD_0002736,1,0,0,9500,9500,0,Pembayaran dibebaskan,Selesai,AugustSales2025.xlsx
2928,ORD_0002929,3,0,0,0,3500,0,Online Payment,Selesai,AugustSales2025.xlsx
3299,ORD_0003300,2,0,0,0,3500,0,COD (Bayar di Tempat),Selesai,AugustSales2025.xlsx
4118,ORD_0004119,1,0,0,10000,10000,0,Pembayaran dibebaskan,Selesai,DecemberSales2024.xlsx
4378,ORD_0004379,1,0,0,9500,9500,0,Pembayaran dibebaskan,Selesai,DecemberSales2024.xlsx


In [59]:
print("Payment Methods:")
print(completed_zero_payment["payment_method"].value_counts(dropna=False))

print("\nDiscount Statistics:")
print(completed_zero_payment["total_discount"].describe())

print("\nShipping Cost Statistics:")
print(
    completed_zero_payment[
        [
            "buyer_shipping_cost",
            "estimated_shipping_discount",
            "estimated_shipping_cost"
        ]
    ].describe()
)

Payment Methods:
payment_method
Pembayaran dibebaskan    17
COD (Bayar di Tempat)    15
Saldo ShopeePay          11
Online Payment            6
SeaBank Bayar Instan      6
SPayLater                 6
Kartu Kredit/Debit        1
Name: count, dtype: int64

Discount Statistics:
count    62.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: total_discount, dtype: float64

Shipping Cost Statistics:
       buyer_shipping_cost  estimated_shipping_discount  \
count                 62.0                    62.000000   
mean                   0.0                  2901.612903   
std                    0.0                  4713.791831   
min                    0.0                     0.000000   
25%                    0.0                     0.000000   
50%                    0.0                     0.000000   
75%                    0.0                  8000.000000   
max                    0.0                 16000.000000   

       estimated

In [60]:
completed_zero_payment.groupby("source_file").size().sort_values(
    ascending=False
)

source_file
NovemberSales2025.xlsx     8
JanuarySales2025.xlsx      6
SeptemberSales2024.xlsx    5
DecemberSales2024.xlsx     5
MaySales2025.xlsx          4
FebruarySales2025.xlsx     4
OctoberSales2025.xlsx      4
JulySales2024.xlsx         4
SeptemberSales2025.xlsx    3
AugustSales2024.xlsx       3
NovemberSales2024.xlsx     3
MarchSales2025.xlsx        3
AugustSales2025.xlsx       3
OctoberSales2024.xlsx      2
AprilSales2024.xlsx        1
AprilSales2025.xlsx        1
JulySales2025.xlsx         1
JuneSales2025.xlsx         1
MaySales2024.xlsx          1
dtype: int64

#### Zero-Payment Order Investigation Result

The investigation identified **62 completed orders with `total_payment = 0`**.

Further examination shows that these transactions are not concentrated in a single period. They are distributed across multiple monthly source files, indicating that the zero-payment cases are not caused by an isolated issue in one specific source file.

Several important patterns were identified:

- All 62 transactions have `total_discount = 0`.
- All 62 transactions have `buyer_shipping_cost = 0`.
- The `estimated_shipping_cost` remains positive, with an average of approximately **15,348**.
- Some transactions contain an `estimated_shipping_discount`, while others do not.
- The transactions involve multiple payment methods rather than a single payment method.
- No negative monetary values or other direct numerical inconsistencies were detected.

Based on the available variables, there is **insufficient evidence to conclude that these records are data errors**.

The zero-payment transactions may represent legitimate but unusual business scenarios, such as payment exemptions, platform promotions, refunds, vouchers, or other payment adjustments that are not fully represented by the available variables.

Therefore, these records should not be automatically removed or modified. Instead, they will be retained and flagged so that their behavior can be analyzed separately without altering the original transaction information.

---------------------------

### Zero-Payment Order Treatment

In [62]:
df["zero_payment_flag"] = (
    (df["order_status_group"] == "Completed") &
    (df["total_payment"] == 0)
).astype(int)

df["zero_payment_flag"].value_counts()

zero_payment_flag
0    20786
1       62
Name: count, dtype: int64

In [63]:
print(
    "Flagged completed zero-payment orders:",
    df["zero_payment_flag"].sum()
)

Flagged completed zero-payment orders: 62


#### Zero-Payment Order Treatment

The investigation identified **62 completed orders with a total payment of zero**.

These records were distributed across multiple source files and payment methods rather than being concentrated in a single period or data source. No negative monetary values or other direct numerical inconsistencies were identified.

Although these transactions are unusual, the available variables do not provide sufficient evidence to classify them as data errors. Zero-payment transactions may represent legitimate platform-specific scenarios such as payment exemptions, promotional mechanisms, refunds, or other adjustments that are not fully captured by the available variables.

Therefore, the records were **retained rather than removed or imputed**.

A binary `zero_payment_flag` was created to identify these transactions for further analysis while preserving the original transaction values.

This approach avoids introducing unsupported assumptions into the dataset while allowing unusual payment behavior to be analyzed separately during exploratory analysis.

-------------------------

### Investigate Completed Orders with Returned Items

In [64]:
completed_with_returns = df[
    (df["order_status_group"] == "Completed") &
    (df["total_returned_quantity"] > 0)
].copy()

print("Total completed orders with returns:", len(completed_with_returns))

completed_with_returns[
    [
        "order_id",
        "total_quantity",
        "total_returned_quantity",
        "total_payment",
        "order_status",
        "payment_method",
        "product_categories",
        "source_file"
    ]
].head(20)

Total completed orders with returns: 157


,order_id,total_quantity,total_returned_quantity,total_payment,order_status,payment_method,product_categories,source_file
37,ORD_0000038,2,1,130540,Selesai,COD (Bayar di Tempat),Plastik / Wadah Plastik,AprilSales2024.xlsx
236,ORD_0000237,2,2,0,Selesai,Saldo ShopeePay,Gantungan Baju / Hanger,AprilSales2024.xlsx
703,ORD_0000704,9,2,122318,Selesai,Saldo ShopeePay,"Gantungan Baju / Hanger, Mangkok Sambal / Saus...",AprilSales2025.xlsx
791,ORD_0000792,1,1,17600,Selesai,Online Payment,Tempat Nasi,AprilSales2025.xlsx
854,ORD_0000855,3,1,19566,Selesai,Saldo ShopeePay,Aksesoris Pintu,AprilSales2025.xlsx
900,ORD_0000901,1,1,0,Selesai,Online Payment,Stempel / Alat Kantor,AprilSales2025.xlsx
1051,ORD_0001052,15,4,130510,Selesai,Online Payment,Piring,AprilSales2025.xlsx
1073,ORD_0001074,20,4,211400,Selesai,SPayLater,Lunch Box / Rantang,AprilSales2025.xlsx
1172,ORD_0001173,2,1,17840,Selesai,SPayLater,Celengan,AprilSales2025.xlsx
1211,ORD_0001212,6,3,69371,Selesai,COD (Bayar di Tempat),"Botol / Gelas / Mug, Mangkok Sambal / Saus",AprilSales2025.xlsx


**Check quantity vs returned quantity relationship**

In [65]:
completed_with_returns[
    ["total_quantity", "total_returned_quantity"]
].describe()

,total_quantity,total_returned_quantity
count,157.000000,157.000000
mean,6.407643,2.356688
std,19.062635,5.888943
min,1.000000,1.000000
25%,1.000000,1.000000
50%,3.000000,1.000000
75%,5.000000,2.000000
max,200.000000,70.000000


In [66]:
invalid_return_qty = completed_with_returns[
    completed_with_returns["total_returned_quantity"]
    > completed_with_returns["total_quantity"]
]

print(
    "Returns exceeding purchased quantity:",
    len(invalid_return_qty)
)

Returns exceeding purchased quantity: 0


**Saperate Full Return vs Partial Return**

In [67]:
completed_with_returns["return_type"] = np.where(
    completed_with_returns["total_returned_quantity"]
    == completed_with_returns["total_quantity"],
    "Full Return",
    "Partial Return"
)

completed_with_returns["return_type"].value_counts()

return_type
Partial Return    87
Full Return       70
Name: count, dtype: int64

In [68]:
return_type_summary = (
    completed_with_returns["return_type"]
    .value_counts()
    .rename_axis("return_type")
    .reset_index(name="orders")
)

return_type_summary["percentage"] = (
    return_type_summary["orders"]
    / return_type_summary["orders"].sum()
    * 100
).round(2)

return_type_summary

,return_type,orders,percentage
0,Partial Return,87,55.41
1,Full Return,70,44.59


**Check relationship with month**

In [71]:
completed_with_returns.groupby(
    "source_file"
).size().sort_values(ascending=False)

source_file
MaySales2025.xlsx          15
FebruarySales2025.xlsx     12
OctoberSales2024.xlsx      12
JuneSales2025.xlsx         10
AprilSales2025.xlsx        10
AugustSales2024.xlsx       10
OctoberSales2025.xlsx      10
NovemberSales2025.xlsx      9
SeptemberSales2025.xlsx     9
DecemberSales2024.xlsx      9
SeptemberSales2024.xlsx     8
JanuarySales2025.xlsx       8
AugustSales2025.xlsx        8
MarchSales2025.xlsx         6
NovemberSales2024.xlsx      5
JulySales2025.xlsx          5
MaySales2024.xlsx           4
AprilSales2024.xlsx         2
MarchSales2024.xlsx         2
FebruarySales2024.xlsx      1
JuneSales2024.xlsx          1
JulySales2024.xlsx          1
dtype: int64

In [72]:
completed_with_returns[
    "product_categories"
].value_counts().head(15)

product_categories
Nampan / Tray                                 14
Celengan                                      10
Baskom / Mangkok Besar                        10
Other                                          9
Lunch Box / Rantang                            7
Aksesoris Pintu                                6
Keranjang                                      6
Mangkok Sambal / Saus                          5
Piring                                         4
Stempel / Alat Kantor                          4
Tempat Nasi                                    3
Toples / Sealware                              2
Botol / Gelas / Mug, Mangkok Sambal / Saus     2
Aksesoris Pintu, Plastik / Wadah Plastik       2
Mangkok Sambal / Saus, Other                   2
Name: count, dtype: int64

#### Completed Order Return Investigation Result

The investigation identified **157 completed orders containing returned items**.

The returned quantities were compared with the original purchased quantities to determine whether these transactions contained logical inconsistencies. No cases were found where the returned quantity exceeded the purchased quantity, indicating that all observed return quantities are logically valid.

Among the 157 returned orders:

- **87 orders (55.41%)** were classified as `Partial Return`.
- **70 orders (44.59%)** were classified as `Full Return`.
- No returned quantity exceeded the corresponding purchased quantity.
- Returned orders were distributed across multiple monthly source files rather than being concentrated in a single period.
- Returned orders also occurred across multiple product categories.

These findings suggest that the presence of returned items in completed orders does not necessarily represent a data quality issue. The `Completed` status may represent the completion of the original order lifecycle, while product returns can occur after the transaction has been completed.

Therefore, there is insufficient evidence to classify these records as erroneous. The returned orders will be retained and represented through additional analytical features for further return behavior analysis.

------------

### Returned Order Treatment

In [73]:
df["has_return"] = (
    df["total_returned_quantity"] > 0
).astype(int)

df["return_type"] = np.select(
    [
        df["total_returned_quantity"] == 0,
        df["total_returned_quantity"] == df["total_quantity"]
    ],
    [
        "No Return",
        "Full Return"
    ],
    default="Partial Return"
)

print("Return Flag:")
print(df["has_return"].value_counts())

print("\nReturn Type:")
print(df["return_type"].value_counts())

Return Flag:
has_return
0    20691
1      157
Name: count, dtype: int64

Return Type:
return_type
No Return         20691
Partial Return       87
Full Return          70
Name: count, dtype: int64


In [74]:
print(
    "Orders with returns:",
    df["has_return"].sum()
)

print(
    "Invalid return quantities:",
    (df["total_returned_quantity"] > df["total_quantity"]).sum()
)

pd.crosstab(
    df["order_status_group"],
    df["return_type"]
)

Orders with returns: 157
Invalid return quantities: 0


return_type,Full Return,No Return,Partial Return
order_status_group,,,
Cancelled,0,2830,0
Completed,70,17772,87
In Progress,0,89,0


#### Returned Order Treatment

The returned-order records were retained because the investigation found no evidence of invalid return quantities or isolated data quality problems.

Rather than removing these transactions, two analytical features were created:

- `has_return`: A binary indicator identifying whether an order contains at least one returned item.
- `return_type`: Classifies each transaction as `No Return`, `Partial Return`, or `Full Return` based on the relationship between purchased and returned quantities.

This treatment preserves the original transaction records while making return behavior easier to analyze during exploratory data analysis and dashboard development.

The resulting features can later support return-rate analysis across product categories, geographic regions, payment methods, shipping options, and other business dimensions.

-----------

## Potential Outlier Investigation

Potential outliers in numerical transaction variables are investigated to identify unusually large or extreme values that may represent data quality issues.

Outliers are not automatically removed because extreme values may represent legitimate transactions, such as bulk purchases, expensive orders, large shipping costs, or high discounts.

The analysis therefore focuses on identifying extreme observations and evaluating their plausibility before applying any treatment.

In [75]:
outlier_columns = [
    "total_quantity",
    "total_weight_gram",
    "total_discount",
    "buyer_shipping_cost",
    "estimated_shipping_discount",
    "estimated_shipping_cost",
    "total_payment"
]

df[outlier_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
total_quantity,20848.0,2.560821,7.796763,1.0,1.0,1.0,2.0,256.0
total_weight_gram,20848.0,2004.529691,7106.357515,10.0,300.0,500.0,1600.0,375000.0
total_discount,20848.0,405.199348,9784.017705,0.0,0.0,0.0,0.0,700000.0
buyer_shipping_cost,20848.0,4189.990695,13578.489678,0.0,0.0,0.0,3000.0,584000.0
estimated_shipping_discount,20848.0,10722.539284,12672.471066,0.0,0.0,9500.0,15000.0,312000.0
estimated_shipping_cost,20848.0,18425.626343,23791.508665,1.0,8000.0,11000.0,20000.0,959200.0
total_payment,20848.0,50683.597036,144019.263221,0.0,13480.0,21800.0,41600.0,3403591.0


In [76]:
outlier_summary = []

for col in outlier_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ]

    outlier_summary.append({
        "column": col,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": len(outliers),
        "outlier_percentage": round(
            len(outliers) / len(df) * 100, 2
        )
    })

outlier_summary = pd.DataFrame(outlier_summary)

outlier_summary

,column,lower_bound,upper_bound,outlier_count,outlier_percentage
0,total_quantity,-0.5,3.5,2614,12.54
1,total_weight_gram,-1650.0,3550.0,2448,11.74
2,total_discount,0.0,0.0,135,0.65
3,buyer_shipping_cost,-4500.0,7500.0,4048,19.42
4,estimated_shipping_discount,-22500.0,37500.0,788,3.78
5,estimated_shipping_cost,-10000.0,38000.0,1815,8.71
6,total_payment,-28700.0,83780.0,2309,11.08


In [77]:
extreme_records = df.nlargest(
    20,
    "total_payment"
)[
    [
        "order_id",
        "total_quantity",
        "total_weight_gram",
        "total_discount",
        "buyer_shipping_cost",
        "estimated_shipping_cost",
        "total_payment",
        "product_categories",
        "order_status_group",
        "source_file"
    ]
]

extreme_records

,order_id,total_quantity,total_weight_gram,total_discount,buyer_shipping_cost,estimated_shipping_cost,total_payment,product_categories,order_status_group,source_file
9706,ORD_0009707,196,196000,0,70000,80000,3403591,Celengan,Completed,JulySales2025.xlsx
19178,ORD_0019179,2,24000,0,344000,384000,3076000,Seal / Baut / Roof,Completed,SeptemberSales2024.xlsx
5830,ORD_0005831,4,52000,0,0,121500,3023425,Seal / Baut / Roof,Completed,FebruarySales2025.xlsx
8956,ORD_0008957,2,24000,0,87700,87700,2889700,Seal / Baut / Roof,Completed,JulySales2024.xlsx
17711,ORD_0017712,8,32000,0,0,150000,2884000,Seal / Baut / Roof,Completed,OctoberSales2025.xlsx
2724,ORD_0002725,2,24000,0,10000,110000,2756000,Seal / Baut / Roof,Completed,AugustSales2025.xlsx
1513,ORD_0001514,2,24000,0,0,87700,2718000,Seal / Baut / Roof,Completed,AugustSales2024.xlsx
13173,ORD_0013174,2,24000,0,0,87700,2703000,Seal / Baut / Roof,Completed,MaySales2024.xlsx
4293,ORD_0004294,2,24000,0,115200,155200,2692454,Seal / Baut / Roof,Completed,DecemberSales2024.xlsx
19633,ORD_0019634,200,60000,700000,106500,206500,2684500,Seal / Baut / Roof,Completed,SeptemberSales2024.xlsx


In [78]:
df.nlargest(
    20,
    "total_quantity"
)[
    [
        "order_id",
        "total_quantity",
        "total_weight_gram",
        "total_payment",
        "product_categories",
        "order_status_group",
        "source_file"
    ]
]

,order_id,total_quantity,total_weight_gram,total_payment,product_categories,order_status_group,source_file
11836,ORD_0011837,256,51200,0,Nampan / Tray,Cancelled,MarchSales2024.xlsx
11837,ORD_0011838,256,51200,1325801,Nampan / Tray,Completed,MarchSales2024.xlsx
12213,ORD_0012214,242,48400,0,Nampan / Tray,Cancelled,MarchSales2025.xlsx
12215,ORD_0012216,242,48400,1403301,Nampan / Tray,Completed,MarchSales2025.xlsx
12234,ORD_0012235,242,48400,1408301,Nampan / Tray,Completed,MarchSales2025.xlsx
12311,ORD_0012312,242,145200,1306661,Nampan / Tray,Completed,MarchSales2025.xlsx
2585,ORD_0002586,200,40000,1002000,Nampan / Tray,Completed,AugustSales2024.xlsx
11318,ORD_0011319,200,150000,2315581,Lunch Box / Rantang,Completed,JuneSales2025.xlsx
19633,ORD_0019634,200,60000,2684500,Seal / Baut / Roof,Completed,SeptemberSales2024.xlsx
9706,ORD_0009707,196,196000,3403591,Celengan,Completed,JulySales2025.xlsx


### Outlier Investigation Result & Treatment

The IQR method identified potential outliers across several numerical transaction variables, including quantity, weight, shipping costs, discounts, and total payment.

However, further inspection of the most extreme transactions showed that these observations remain plausible within the context of the dataset. Large quantities are primarily associated with bulk-order transactions, while high payment values are supported by corresponding product quantities, shipping costs, product categories, and other transaction attributes.

In addition, some variables such as `total_discount` contain a large proportion of zero values. As a result, the IQR can become very small or zero, causing any non-zero value to be classified as a statistical outlier even when the transaction is valid.

No negative monetary values or invalid returned quantities were previously identified. Zero-payment completed orders were also investigated separately and retained with an analytical flag.

Therefore, no outliers were removed, capped, or imputed. The original values were retained to avoid removing potentially valid business transactions and to preserve the natural variation of the transactional data.

------

## Duplicate Validation

Duplicate records are examined to ensure that transactions were not accidentally duplicated during the data collection, merging, or transformation process.

Two types of duplicates are evaluated:

- **Exact duplicates:** Rows where all column values are identical.
- **Duplicate order IDs:** Multiple records sharing the same `order_id`, which may indicate duplicated transactions or require further investigation.

Duplicate records are not automatically removed because repeated order IDs may still represent legitimate records depending on the structure of the original data.

**Exact duplicate**

In [79]:
exact_duplicates = df.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)

Exact duplicate rows: 0


**Duplicate order_id**

In [80]:
duplicate_order_ids = df["order_id"].duplicated(keep=False)

print(
    "Rows with duplicated order IDs:",
    duplicate_order_ids.sum()
)

print(
    "Unique duplicated order IDs:",
    df.loc[duplicate_order_ids, "order_id"].nunique()
)

Rows with duplicated order IDs: 0
Unique duplicated order IDs: 0


In [81]:
df.loc[
    duplicate_order_ids
].sort_values("order_id").head(30)

,order_id,total_quantity,total_weight_gram,total_returned_quantity,total_discount,product_categories,num_product_categories,order_status,cancellation_reason,shipping_option,payment_method,city_regency,province,buyer_shipping_cost,estimated_shipping_discount,total_payment,estimated_shipping_cost,order_created_at,source_file,source_month_name,source_year,source_month,order_year,order_month,year_month,timestamp_available,order_status_raw,order_status_group,zero_payment_flag,has_return,return_type


### Duplicate Validation Result

No duplicate records were identified in the dataset.

- **Exact duplicate rows:** 0
- **Rows with duplicated `order_id`:** 0
- **Duplicated unique `order_id`:** 0

Each `order_id` uniquely represents a single transaction, and no records were duplicated during the data collection, merging, or transformation process.

Therefore, no duplicate removal was required.

------------

## Final Data Quality Check

A final data quality assessment is performed after all cleaning and transformation steps to verify that the dataset is ready for exploratory data analysis, SQL analysis, and dashboard development.

The final validation checks dataset dimensions, missing values, duplicate records, data types, categorical consistency, and the analytical features created during the cleaning process.

In [82]:
print("FINAL DATASET QUALITY CHECK")
print("=" * 50)

print("\nDataset Shape:")
print(df.shape)

print("\nTotal Records:")
print(len(df))

print("\nTotal Columns:")
print(df.shape[1])

print("\nExact Duplicate Rows:")
print(df.duplicated().sum())

print("\nDuplicate Order IDs:")
print(df["order_id"].duplicated().sum())

print("\nMissing Values:")
print(df.isna().sum())

print("\nData Types:")
print(df.dtypes)

FINAL DATASET QUALITY CHECK

Dataset Shape:
(20848, 31)

Total Records:
20848

Total Columns:
31

Exact Duplicate Rows:
0

Duplicate Order IDs:
0

Missing Values:
order_id                          0
total_quantity                    0
total_weight_gram                 0
total_returned_quantity           0
total_discount                    0
product_categories                0
num_product_categories            0
order_status                      0
cancellation_reason               0
shipping_option                   0
payment_method                    0
city_regency                      0
province                          0
buyer_shipping_cost               0
estimated_shipping_discount       0
total_payment                     0
estimated_shipping_cost           0
order_created_at               1980
source_file                       0
source_month_name                 0
source_year                       0
source_month                      0
order_year                        0
order_mon

## Final Data Cleaning Summary

The data cleaning and preparation process has been completed successfully.

The final dataset contains **20,848 transactions and 31 variables**, with no exact duplicate records or duplicated `order_id` values.

The cleaning process included:

- Standardizing column names and correcting data types.
- Investigating and handling missing values based on their business context.
- Preserving missing `order_created_at` values when the original transaction timestamp could not be reliably recovered.
- Recovering month and year information from source files for transactions without timestamps.
- Standardizing categorical variables such as order status, payment method, province, and city/regency.
- Validating transaction quantities and monetary values using business-rule checks.
- Investigating completed orders with zero payment and retaining them using `zero_payment_flag`.
- Investigating returned transactions and creating `has_return` and `return_type` analytical features.
- Investigating statistical outliers and retaining plausible extreme transactions.
- Validating duplicate records and confirming that each `order_id` is unique.
- Creating additional analytical features to support exploratory analysis and dashboard development.

The remaining **1,980 missing values in `order_created_at` are intentionally retained** because the exact transaction timestamps cannot be reliably reconstructed. Their month and year information remains available through the source-derived temporal features.

The cleaned dataset is considered suitable for exploratory data analysis, SQL analysis, and dashboard development.

---------

## Export Clean Dataset

In [83]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "cleaned_ecommerce_data.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Cleaned dataset saved to: {output_path}")
print(f"Final dataset shape: {df.shape}")

Cleaned dataset saved to: ..\data\processed\cleaned_ecommerce_data.csv
Final dataset shape: (20848, 31)
